In [6]:
import pandas as pd

RECORDS = "records.csv"

# This file contains repeated runs for some tasks (task_id 54-131 appear twice).
# Rows are laid out in blocks: one `full` row followed by its compressed configs,
# so each block is one run of one task and pairing is done within a block.
# Repeats are dropped: only the first run of each task is kept.

records = pd.read_csv(RECORDS)

# Block id: a new block starts at every `full` row.
records["run_id"] = (records["kv_caching"] == "full").cumsum()

# Keep only the first run of each task (lowest run_id = earliest block in the file).
first_run = records.groupby("task_id")["run_id"].min()
records = records[records["run_id"] == records["task_id"].map(first_run)].copy()

# 1) delta_H_seq = H_seq(compressed) - H_seq(full) for the same task in the same run.
h_full = (
    records.loc[records["kv_caching"] == "full", ["run_id", "task_id", "H_seq"]]
    .rename(columns={"H_seq": "H_seq_full"})
)
records = records.merge(h_full, on=["run_id", "task_id"], how="left", validate="many_to_one")
records["delta_H_seq"] = records["H_seq"] - records["H_seq_full"]

# 2) Drop the full-KV rows; only compressed configs carry a delta / KL.
compressed = records[records["kv_caching"] != "full"].copy()

# 3) Average across tasks within each compression ratio.
per_ratio = (
    compressed.groupby("ratio")
    .agg(
        delta_H_seq_mean=("delta_H_seq", "mean"),
        delta_H_seq_std=("delta_H_seq", "std"),
        kl_seq_mean=("kl_seq", "mean"),
        kl_seq_std=("kl_seq", "std"),
        n_tasks=("task_id", "nunique"),
    )
    .reset_index()
)

per_ratio

,ratio,delta_H_seq_mean,delta_H_seq_std,kl_seq_mean,kl_seq_std,n_tasks
0,0.25,4.138230,10.569299,4.379253,3.909614,150
1,0.50,7.740914,12.382037,6.182192,5.424361,150
2,0.75,13.251486,12.123827,9.911172,8.583183,150
3,0.95,31.262826,15.545080,20.100285,11.176445,150
